# 개별종목 조합A — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합A 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합A의 피처 값만 지정합니다.
import json

COMBINATION = 'A'
FEATURE_COLUMNS = (
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'macd_hist_ratio',
    'bb_bandwidth',
    'bb_position',
    'atr_ratio',
    'hv_20',
    'vol_ratio_20',
    'obv_slope_20',
    'daily_return',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합A 피처: ('sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'macd_hist_ratio', 'bb_bandwidth', 'bb_position', 'atr_ratio', 'hv_20', 'vol_ratio_20', 'obv_slope_20', 'daily_return', 'five_day_return')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4946,0.5012,-0.0066,0.3188,0.3610,0.0738,0.3844,0.0849,0.1772
1,2,balanced,980,20150123,20150421,0.3950,0.3978,-0.0029,0.3667,0.3737,0.0675,0.3811,0.2627,0.3309
2,3,balanced,1210,20151228,20160328,0.3659,0.3762,-0.0103,0.3651,0.3654,0.0511,0.3854,0.3444,0.3582
3,4,balanced,1439,20161202,20170228,0.4603,0.4617,-0.0014,0.3694,0.3855,0.0989,0.4103,0.1550,0.2648
4,5,balanced,1669,20171113,20180207,0.4096,0.3901,0.0195,0.3834,0.3904,0.0928,0.4051,0.2852,0.3506
5,6,balanced,1899,20181024,20190118,0.4071,0.3725,0.0346,0.4036,0.4123,0.1229,0.4264,0.4309,0.4135
6,7,balanced,2129,20190930,20191224,0.4665,0.4781,-0.0116,0.3669,0.3840,0.0989,0.4109,0.2254,0.3224
7,8,balanced,2359,20200902,20201130,0.3895,0.3476,0.0418,0.3891,0.3940,0.0910,0.4000,0.4480,0.4071
8,9,balanced,2589,20210806,20211105,0.3808,0.3914,-0.0106,0.3683,0.3895,0.0794,0.3846,0.2252,0.3067
9,10,balanced,2818,20220714,20221012,0.3634,0.3454,0.0179,0.3627,0.3699,0.0550,0.3838,0.2759,0.3284


,OOS 폴드 평균
accuracy,0.4097
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0128
macro_f1,0.3725
balanced_accuracy,0.3835
mcc,0.0834
pr_auc_macro_ovr,0.3974
down_recall,0.2942
core_harmonic_mean,0.3370


재실행 명령: python scripts/run_stock_model_experiment.py
